In [1]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import logging, tqdm
import dask.array as da
import zarr
from GroupRegNet.model import regnet, loss, util
from GroupRegNet.utils import structure

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# --- Parameter ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
zarr_path = "/media/shooty/Dev/repos/MRI-MoCoCo/MRI-Datasets/DCE"
output_path = "./groupregnet_dce_output"
os.makedirs(output_path, exist_ok=True)

# --- Konfig ---
config = dict(
    dim = 3,
    scale = 0.5,
    initial_channels = 32,
    depth = 4,
    normalization = True,
    learning_rate = 1e-2,
    smooth_reg = 1e-3,
    cyclic_reg = 1e-2,
    max_num_iteration = 1000,
    ncc_window_size = 5,
    stop_std = 0.0007,
    stop_query_len = 50,
    batch_size = 8,  # Chunkgröße
)
config = structure.Struct(**config)

# --- Lade Zarr-Daten mit Dask ---
data = da.from_zarr(zarr_path)  # (X, Y, Z, T)
data = da.transpose(data, (3, 2, 1, 0))  # (T, Z, Y, X)

n_timepoints = data.shape[0]
res_list = []

# --- Training pro Chunk (Batch) ---
for t_start in range(0, n_timepoints, config.batch_size):
    t_end = min(t_start + config.batch_size, n_timepoints)
    logging.info(f"Lade Chunk: T={t_start} bis T={t_end}")
    
    data_chunk = data[t_start:t_end].compute()  # -> (B, Z, Y, X)
    data_chunk = data_chunk[:, None, :, :, :]  # -> (B, 1, Z, Y, X)
    input_image = torch.from_numpy(data_chunk.astype(np.float32)).to(device)

    # --- Normalisierung pro Chunk ---
    input_image = (input_image - input_image.mean()) / input_image.std()

    # --- Modellinstanz pro Batch ---
    regnet_model = regnet.RegNet_single(
        dim=config.dim,
        n=input_image.shape[0],
        scale=config.scale,
        depth=config.depth,
        initial_channels=config.initial_channels,
        normalization=config.normalization
    ).to(device)

    ncc_loss = loss.NCC(config.dim, config.ncc_window_size).to(device)
    optimizer = torch.optim.Adam(regnet_model.parameters(), lr=config.learning_rate)
    stop_criterion = util.StopCriterion(config.stop_std, config.stop_query_len)

    # --- Training pro Chunk ---
    pbar = tqdm.tqdm(range(config.max_num_iteration))
    for i in pbar:
        optimizer.zero_grad()
        res = regnet_model(input_image)

        # Ähnlichkeitsverlust
        simi_loss = ncc_loss(res['warped_input_image'], res['template'])
        total_loss = simi_loss

        # Glättung
        if config.smooth_reg > 0:
            smooth_loss = loss.smooth_loss(res['scaled_disp_t2i'], res['scaled_template'])
            total_loss += config.smooth_reg * smooth_loss
        else:
            smooth_loss = torch.tensor(0.0)

        # Zyklische Konsistenz
        if config.cyclic_reg > 0 and 'disp_i2t' in res:
            cyclic_loss = torch.mean((torch.sum(res['scaled_disp_t2i'], 0))**2)**0.5
            total_loss += config.cyclic_reg * cyclic_loss
        else:
            cyclic_loss = torch.tensor(0.0)

        total_loss.backward()
        optimizer.step()

        stop_criterion.add(simi_loss.item())
        pbar.set_description(f'[{t_start}-{t_end}] {i}: simi={simi_loss.item():.4f}, smooth={smooth_loss.item():.4f}, cyclic={cyclic_loss.item():.4f}')
        if stop_criterion.stop():
            break

    # --- Ergebnisse speichern ---
    res_np = res['warped_input_image'].detach().cpu().numpy()
    np.save(os.path.join(output_path, f"warped_dce_chunk_{t_start}_{t_end}.npy"), res_np)

    # Optional: Zarr
    z = zarr.open(os.path.join(output_path, f"warped_dce_chunk_{t_start}_{t_end}.zarr"), mode="w", shape=res_np.shape, dtype='float32')
    z[:] = res_np

    # Optional: Log der Loss-Kurve
    plt.figure()
    plt.plot(stop_criterion.loss_list)
    plt.title(f"Loss für Chunk {t_start}-{t_end}")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.savefig(os.path.join(output_path, f"loss_curve_{t_start}_{t_end}.png"), dpi=300)
    plt.close()

print("Registrierung abgeschlossen.")


INFO: Lade Chunk: T=0 bis T=8
  0%|          | 0/1000 [00:00<?, ?it/s]/home/shooty/anaconda3/envs/ds-env-04/lib/python3.13/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
INFO: 
Add grid shape (50, 256, 256)
[0-8] 461: simi=-0.9994, smooth=0.7086, cyclic=0.0000:  46%|████▌     | 461/1000 [04:29<05:14,  1.71it/s]
INFO: Lade Chunk: T=8 bis T=16
  0%|          | 0/1000 [00:00<?, ?it/s]INFO: 
Add grid shape (50, 256, 256)
[8-16] 327: simi=-0.9985, smooth=0.7565, cyclic=0.0000:  33%|███▎      | 327/1000 [03:11<06:33,  1.71it/s]
INFO: Lade Chunk: T=16 bis T=24
  0%|          | 0/1000 [00:00<?, ?it/s]INFO: 
Add grid shape (50, 256, 256)
[16-24] 999: simi=-1.3135, smooth=3.1724, cyclic=0.0000: 100%|██████████| 1000/1000 [09:42<00:00,  1.72it/s]
INFO: Lad

KeyboardInterrupt: 